# Phase 2: Package Prioritization and Box Filling

**Student:** Tanishq  
Sort parcels, then use 0/1 knapsack dynamic programming for one vehicle trip.

## Theory

Parcels are sorted by higher priority, then earlier deadline. Python Timsort is stable.

The 0/1 knapsack recurrence is:

`dp[i][c] = max(dp[i-1][c], benefit[i] + dp[i-1][c-weight[i]])` when the parcel fits.

Time and space are `O(n × capacity)`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from app import format_dp_table, load_parcels
from src.knapsack import knapsack_select_parcels
from src.sorting import parcel_benefit, sort_parcels_by_priority_and_deadline

VEHICLE_CAPACITY = 12
parcels = load_parcels(ROOT / "datasets/parcels.csv")
sorted_parcels = sort_parcels_by_priority_and_deadline(parcels)

print("Sorted parcels (priority, then earliest deadline):")
for parcel in sorted_parcels:
    print(
        f"{parcel.parcel_id} | dest={parcel.destination} | {parcel.priority:<6} | "
        f"deadline={parcel.deadline} | {parcel.weight} kg | benefit={parcel_benefit(parcel)}"
    )

In [ ]:
maximum_benefit, selected_parcels, dp_table = knapsack_select_parcels(sorted_parcels, VEHICLE_CAPACITY)
total_weight = sum(parcel.weight for parcel in selected_parcels)
print(f"Vehicle capacity: {VEHICLE_CAPACITY} kg")
print("Selected parcels:", ", ".join(parcel.parcel_id for parcel in selected_parcels))
print(f"Total weight: {total_weight} kg")
print(f"Optimal benefit: {maximum_benefit}")
print(f"Unused capacity: {VEHICLE_CAPACITY - total_weight} kg")
print()
print("\n".join(format_dp_table(sorted_parcels, dp_table)))

In [ ]:
assert total_weight <= VEHICLE_CAPACITY
assert maximum_benefit == 550
print("Normal case passed.")

small_benefit, small_selection, _ = knapsack_select_parcels(sorted_parcels, 3)
assert small_benefit == 185
assert sum(parcel.weight for parcel in small_selection) <= 3
print("Small-capacity edge case passed.")

sorted_ids = [parcel.parcel_id for parcel in sorted_parcels]
assert sorted_ids.index("P04") < sorted_ids.index("P09")
print("Stable tie case passed: P04 remains before P09.")

## Phase 2 summary

The 12 kg vehicle is packed with `P02, P01, P04, P09, P06` for benefit 550. The DP table is the packing evidence required by the rubric.